<a id="sec0"></a>
# Ejercicio integrador · ¿El sentimiento de las noticias ayuda a pronosticar la volatilidad?


La volatilidad es la materia prima de la gestión de riesgos: con ella se calcula el VaR, se valoran
opciones, se fijan márgenes y se decide cuánto capital asignar a cada posición. Los modelos clásicos,
como GARCH, la pronostican solo con la historia de los precios. Pero los precios reaccionan a
información, y buena parte de esa información llega en forma de texto: titulares, reportes de
analistas, comunicados.

En este ejercicio vas a responder, con datos y con una prueba estadística, una pregunta concreta:

> **¿Un índice diario de sentimiento de noticias, calculado con FinBERT, mejora el pronóstico de
> volatilidad frente a un modelo que solo usa datos de mercado?**

Para responderla vas a comparar tres modelos sobre el mismo periodo de prueba:

| # | Modelo | Qué información usa |
|---|---|---|
| 1 | **GARCH(1,1)** (línea base) | Solo retornos pasados |
| 2 | **LSTM de mercado** | Retornos rezagados, volatilidad histórica y volumen |
| 3 | **LSTM de mercado + sentimiento** | Todo lo anterior + índice diario de sentimiento FinBERT |

La comparación clave es **2 contra 3**: son la misma red, entrenada de la misma forma, y la única
diferencia es si recibe o no las variables de sentimiento. Si el modelo 3 no pronostica mejor que el
2, el sentimiento no está agregando información útil (al menos no de la forma en que lo construimos).

### La empresa: Occidental Petroleum (OXY)

La idea original era usar Apple (AAPL). Al revisar el dataset de noticias encontramos que **limita la
cantidad de titulares por empresa**, y para las más grandes (AAPL, AMZN, TSLA, FB) solo conserva los
de los últimos meses: AAPL no tiene **ningún** titular entre 2012 y 2018. Una LSTM no puede aprender a
usar el sentimiento si en el entrenamiento siempre vale cero.

Usamos entonces **OXY**, una petrolera que sí tiene noticias de forma continua en los tres periodos.
Además, su periodo de prueba es especialmente didáctico: incluye el desplome de marzo de 2020
(OXY cayó **52% en un solo día**, el 9 de marzo, por la guerra de precios del petróleo y el COVID)
y el precio negativo del crudo WTI en abril de 2020. En la sección 3 verás cómo revisar la cobertura
de cualquier otra empresa, por si quieres repetir el ejercicio con otra.

### Protocolo de evaluación

| Periodo | Fechas | Para qué se usa |
|---|---|---|
| Entrenamiento | 2012-01-03 a 2017-12-29 | Estimar GARCH y ajustar los pesos de las LSTM |
| Validación | 2018-01-02 a 2018-12-31 | Detener el entrenamiento de las LSTM a tiempo (*early stopping*) |
| Prueba | 2019-01-02 a 2020-05-29 | Comparar los tres modelos. **Solo se mira al final** |

La fecha final es conservadora: el dataset de noticias se publicó a comienzos de julio de 2020, así
que no hay riesgo de que falten noticias de los últimos días por cortes en la recolección.



### 0.1 Entorno de ejecución

El cuaderno está pensado para **Google Colab** y funciona en CPU. Con GPU (T4) el cálculo de
sentimiento con FinBERT pasa de unos minutos a unos segundos, pero no es indispensable.

In [ ]:
# Instalación de librerías con versiones fijadas.
#
# No reinstalamos `torch` ni `tensorflow`: Colab ya los trae compilados contra
# su GPU, y reemplazarlos con pip puede romper el soporte de GPU.
#
# `yfinance` es la única excepción a fijar versión: Yahoo cambia su API con
# frecuencia y las versiones viejas dejan de funcionar. Siempre instalamos la
# más reciente.
!pip install -q \
    "transformers==4.46.3" \
    "arch==7.2.0" \
    "kagglehub==1.0.2" \
    "scikit-learn==1.5.2" \
    "pandas==2.2.3" \
    "matplotlib==3.9.2" \
    "seaborn==0.13.2"
!pip install -q --upgrade yfinance

In [ ]:
import os

# Evita que `transformers` intente cargar TensorFlow: usamos TensorFlow para
# las LSTM y PyTorch para FinBERT, y así cada librería usa su propio motor.
os.environ["USE_TF"] = "0"

import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import yfinance as yf
from arch import arch_model
from sklearn.preprocessing import StandardScaler

import torch
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from IPython.display import display, Markdown

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (11, 4)
pd.set_option("display.max_colwidth", 110)

print(f"TensorFlow {tf.__version__} | PyTorch {torch.__version__}")
print(f"GPU para FinBERT: {'sí' if torch.cuda.is_available() else 'no (usaremos CPU)'}")

### 0.2 Parámetros del ejercicio

Todas las decisiones de diseño están en la siguiente celda. Si quieres repetir el ejercicio con otra
empresa, otro horizonte o más semillas, este es el único lugar que tienes que cambiar.

In [ ]:
TICKER = "OXY"

# Periodos del protocolo de evaluación
INI_TRAIN, FIN_TRAIN = "2012-01-03", "2017-12-29"
INI_VAL, FIN_VAL = "2018-01-02", "2018-12-31"
INI_TEST, FIN_TEST = "2019-01-02", "2020-05-29"

# Descargamos precios desde un año antes para poder calcular volatilidades
# históricas y ventanas de la LSTM desde el primer día de entrenamiento, y hasta
# un mes después del final para tener los retornos que definen el objetivo.
INI_DESCARGA, FIN_DESCARGA = "2011-01-01", "2020-07-01"

H = 5          # horizonte: pronosticamos la volatilidad de los próximos H días hábiles
L = 22         # longitud de la ventana de la LSTM (≈ un mes hábil)
N_SEMILLAS = 5 # cuántas veces entrenamos cada LSTM con semillas distintas

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
keras.utils.set_random_seed(SEED)

<a id="sec1"></a>
## 1. Datos de mercado

Descargamos precios diarios ajustados (por dividendos y *splits*) y volumen desde Yahoo Finance.
Trabajamos con **retornos logarítmicos en porcentaje**, $r_t = 100 \cdot \ln(P_t / P_{t-1})$: la
escala en porcentaje evita problemas numéricos al estimar GARCH, y el logaritmo hace que los retornos
de varios días se sumen.

In [ ]:
precios = yf.download(TICKER, start=INI_DESCARGA, end=FIN_DESCARGA, auto_adjust=True, progress=False)

# Las versiones recientes de yfinance devuelven columnas con dos niveles
# (variable, ticker) incluso para un solo ticker; nos quedamos con el primero.
if isinstance(precios.columns, pd.MultiIndex):
    precios.columns = precios.columns.get_level_values(0)

mercado = precios[["Close", "Volume"]].rename(columns={"Close": "precio", "Volume": "volumen"})
mercado["ret"] = 100 * np.log(mercado["precio"]).diff()
mercado = mercado.dropna()

print(f"{len(mercado)} días hábiles, de {mercado.index.min().date()} a {mercado.index.max().date()}")
mercado.loc["2020-03-05":"2020-03-12"]

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].plot(mercado.index, mercado["precio"])
axes[0].set_ylabel("Precio ajustado (USD)")
axes[0].set_title(f"{TICKER}: precio y retornos diarios")
axes[1].plot(mercado.index, mercado["ret"], linewidth=0.6)
axes[1].set_ylabel("Retorno diario (%)")
for ax in axes:
    ax.axvspan(pd.Timestamp(INI_VAL), pd.Timestamp(FIN_VAL), color="orange", alpha=0.12, label="Validación")
    ax.axvspan(pd.Timestamp(INI_TEST), pd.Timestamp(FIN_TEST), color="red", alpha=0.10, label="Prueba")
axes[0].legend(loc="upper right")
plt.tight_layout()
plt.show()

Fíjate en dos rasgos típicos de la volatilidad que vamos a intentar pronosticar:

- **Agrupamiento**: los días de movimientos grandes tienden a llegar juntos (2015-2016, con la caída
  del petróleo; marzo-abril de 2020).
- **Cambios de régimen**: la volatilidad del periodo de prueba es muy distinta a la del entrenamiento.
  Ningún modelo vio en entrenamiento un día como el −52% del 9 de marzo de 2020. Esto es lo que hace
  la prueba tan exigente... y tan interesante.

<a id="sec2"></a>
## 2. Variable objetivo: volatilidad realizada futura

La volatilidad no se observa directamente: hay que medirla. Con datos diarios, una medida estándar es
la **volatilidad realizada** de los próximos $H$ días:

$$
RV_{t}^{(H)} = \sqrt{\frac{1}{H} \sum_{i=1}^{H} r_{t+i}^2}
$$

Es la raíz del promedio de los retornos al cuadrado entre $t+1$ y $t+H$, expresada en % diario.
Con $H = 5$ pronosticamos la volatilidad de la próxima semana hábil, **usando solo lo que se sabía al
cierre del día $t$**.

¿Por qué no pronosticar directamente $|r_{t+1}|$ (un solo día)? Porque es una medida muy ruidosa: un
día tranquilo en medio de un periodo turbulento haría parecer que el modelo se equivocó mucho. Promediar
$H$ días reduce ese ruido sin perder la dinámica.

La LSTM va a predecir $\ln RV$ en lugar de $RV$: el logaritmo hace la distribución mucho más simétrica
(la volatilidad tiene una cola derecha muy larga) y garantiza que el pronóstico, al deshacer el
logaritmo, sea siempre positivo.

In [ ]:
# rolling(H).mean() en el día s promedia r² de s-H+1 a s; al desplazarlo H días
# hacia atrás, en el día t queda el promedio de r² de t+1 a t+H.
mercado["rv_futura"] = np.sqrt((mercado["ret"] ** 2).rolling(H).mean().shift(-H))
mercado["log_rv_futura"] = np.log(mercado["rv_futura"])

# Fecha del último retorno que entra en el objetivo de cada día. La usamos más
# adelante para que ninguna muestra de un periodo "vea" retornos del siguiente.
mercado["fecha_fin_objetivo"] = pd.Series(mercado.index, index=mercado.index).shift(-H)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
(mercado["rv_futura"] * np.sqrt(252)).plot(ax=axes[0], linewidth=0.7)
axes[0].set_title(f"Volatilidad realizada de los próximos {H} días (anualizada, %)")
sns.histplot(mercado["log_rv_futura"].dropna(), ax=axes[1], bins=60)
axes[1].set_title("Distribución de ln(RV): mucho más simétrica")
plt.tight_layout()
plt.show()

<a id="sec3"></a>
## 3. Noticias: el dataset *Daily Financial News for 6000+ Stocks*

### 3.1 Descarga

Usamos el dataset público de Kaggle **Daily Financial News for 6000+ Stocks** (Miguel Aenlle, 2020),
con ~4 millones de titulares de 2009 a 2020 para más de 6 000 empresas de EE. UU. Licencia **CC0
(dominio público)**. Versión 2, publicada el 4 de julio de 2020.

- **Página del dataset:**
  [kaggle.com/datasets/miguelaenlle/massive-stock-news-analysis-db-for-nlpbacktests](https://www.kaggle.com/datasets/miguelaenlle/massive-stock-news-analysis-db-for-nlpbacktests)
  (botón *Download*; requiere una cuenta gratuita de Kaggle).
- **Identificador para descargar desde código:** `miguelaenlle/massive-stock-news-analysis-db-for-nlpbacktests`

| Archivo | Tamaño | Contenido | ¿Lo usamos? |
|---|---|---|---|
| `raw_analyst_ratings.csv` | ~330 MB | Titulares de Benzinga (noticias y cambios de recomendación de analistas), con URL, medio, fecha y ticker | Sí |
| `raw_partner_headlines.csv` | ~400 MB | Titulares de medios aliados de Benzinga (Seeking Alpha, Zacks, GuruFocus...), mismas columnas | Sí |
| `analyst_ratings_processed.csv` | ~160 MB | Versión reducida de `raw_analyst_ratings.csv` (solo titular, fecha y ticker) | No: repite la información |

En el repositorio del curso los archivos están en la carpeta **`news_archive/`**, junto a este
cuaderno. La siguiente celda los busca en este orden:

1. **Carpeta local `news_archive/`** (si ejecutas el cuaderno desde el repositorio).
2. **Descarga automática con `kagglehub`** (lo normal en Colab; no necesita credenciales para este
   dataset, tarda menos de un minuto y descarga ~730 MB).
3. Si ambas fallan, te indica cómo descargarlos a mano y subirlos a Google Drive.

In [ ]:
ARCHIVOS_NOTICIAS = ["raw_analyst_ratings.csv", "raw_partner_headlines.csv"]
KAGGLE_DATASET = "miguelaenlle/massive-stock-news-analysis-db-for-nlpbacktests"

CARPETA_LOCAL = "news_archive"  # cámbiala si guardaste los archivos en otra carpeta (p. ej. en Drive)

rutas_noticias = {}
if all(os.path.exists(os.path.join(CARPETA_LOCAL, f)) for f in ARCHIVOS_NOTICIAS):
    rutas_noticias = {f: os.path.join(CARPETA_LOCAL, f) for f in ARCHIVOS_NOTICIAS}
    print(f"Usando los archivos locales de la carpeta '{CARPETA_LOCAL}/'.")
else:
    try:
        import kagglehub

        print("No encontré la carpeta local; descargando desde Kaggle con kagglehub...")
        for f in ARCHIVOS_NOTICIAS:
            rutas_noticias[f] = kagglehub.dataset_download(KAGGLE_DATASET, path=f)
        print("Descarga completa.")
    except Exception as error:
        print(f"No se pudo descargar automáticamente ({type(error).__name__}: {error}).")
        print("Plan manual:")
        print(f"  1. Descarga el dataset desde https://www.kaggle.com/datasets/{KAGGLE_DATASET}")
        print("  2. Descomprímelo y sube los dos archivos CSV a una carpeta de Google Drive.")
        print("  3. Monta Drive (from google.colab import drive; drive.mount('/content/drive'))")
        print("     y cambia CARPETA_LOCAL a esa carpeta. Luego vuelve a ejecutar esta celda.")

rutas_noticias

### 3.2 Filtrar los titulares de la empresa

Los archivos son grandes (~3 millones de filas entre ambos). Para no llenar la memoria de Colab los
leemos **por bloques** y nos quedamos solo con las filas de `TICKER`.

In [ ]:
def cargar_titulares(rutas, ticker, tam_bloque=500_000):
    """Lee los CSV por bloques y devuelve solo los titulares del ticker indicado."""
    partes = []
    for nombre, ruta in rutas.items():
        for bloque in pd.read_csv(ruta, usecols=["headline", "publisher", "date", "stock"],
                                  chunksize=tam_bloque):
            seleccion = bloque[bloque["stock"] == ticker].copy()
            seleccion["archivo"] = nombre
            partes.append(seleccion)
    titulares = pd.concat(partes, ignore_index=True)
    # Casi todas las fechas vienen sin hora (00:00:00); nos quedamos solo con el día.
    titulares["fecha"] = pd.to_datetime(titulares["date"].astype(str).str[:10])
    titulares = titulares.drop_duplicates(subset=["headline", "fecha"])
    return titulares.sort_values("fecha").reset_index(drop=True)


inicio = time.perf_counter()
titulares = cargar_titulares(rutas_noticias, TICKER)
print(f"{len(titulares)} titulares únicos de {TICKER} "
      f"({titulares['fecha'].min().date()} a {titulares['fecha'].max().date()}), "
      f"leídos en {time.perf_counter() - inicio:.0f} s")
titulares.sample(8, random_state=SEED)[["fecha", "publisher", "headline"]]

### 3.3 ¿Hay suficientes noticias en cada periodo?

Esta es la revisión que descartó a AAPL. La función `cobertura` cuenta, para cada periodo, cuántos
titulares hay y en qué porcentaje de los días hábiles hubo al menos uno. Puedes usarla para evaluar
cualquier otra empresa antes de cambiar `TICKER`.

In [ ]:
PERIODOS = {
    "entrenamiento": (INI_TRAIN, FIN_TRAIN),
    "validación": (INI_VAL, FIN_VAL),
    "prueba": (INI_TEST, FIN_TEST),
}


def cobertura(titulares, dias_habiles):
    filas = []
    for nombre, (ini, fin) in PERIODOS.items():
        t = titulares[(titulares["fecha"] >= ini) & (titulares["fecha"] <= fin)]
        dias = dias_habiles[(dias_habiles >= ini) & (dias_habiles <= fin)]
        filas.append({
            "periodo": nombre,
            "titulares": len(t),
            "días hábiles": len(dias),
            "% días con noticias": round(100 * t.loc[t["fecha"].isin(dias), "fecha"].nunique() / len(dias), 1),
        })
    return pd.DataFrame(filas).set_index("periodo")


display(cobertura(titulares, mercado.index))

fig, ax = plt.subplots(figsize=(11, 3))
titulares.groupby([titulares["fecha"].dt.year, "archivo"]).size().unstack().plot(kind="bar", stacked=True, ax=ax)
ax.set_title(f"Titulares de {TICKER} por año y archivo de origen")
ax.set_xlabel("")
plt.show()

Dos limitaciones del dataset que conviene tener presentes desde ya:

- **Cobertura irregular.** Hay días sin ninguna noticia; en esos días el índice de sentimiento valdrá
  cero (neutral) y le diremos a la red cuántos titulares hubo para que distinga "neutral" de "sin
  información".
- **Titulares que no son sobre la empresa.** Un titular aparece asignado a un ticker si lo menciona,
  aunque sea de pasada ("Benzinga's Top ETF Gainers (DRN, FAS, EDC, ERX)", "Wall Street Breakfast...").
  Eso agrega ruido al índice. En las actividades finales te proponemos filtrarlos.

<a id="sec4"></a>
## 4. Índice diario de sentimiento con FinBERT

### 4.1 Puntaje de cada titular

Usamos **FinBERT** (`ProsusAI/finbert`), el modelo *encoder* afinado para sentimiento financiero que
vimos en la sesión de NLP. Para cada titular obtenemos las probabilidades de las tres clases y las
resumimos en un solo número:

$$
s = P(\text{positive}) - P(\text{negative}) \in [-1, 1]
$$

Un titular claramente negativo queda cerca de −1, uno positivo cerca de +1 y uno neutral cerca de 0.
Usar la diferencia de probabilidades (en lugar de la etiqueta ganadora) conserva la **intensidad**:
no es lo mismo un titular negativo con 95% de confianza que uno con 45%.

Solo puntuamos los titulares que caen dentro del periodo del ejercicio (más un pequeño margen previo).
En CPU esto toma unos minutos; con GPU, segundos. Al terminar guardamos el resultado en un CSV para no
tener que repetirlo si reinicias el cuaderno.

In [ ]:
from transformers import pipeline

ARCHIVO_PUNTAJES = f"puntajes_finbert_{TICKER}.csv"

titulares_periodo = titulares[
    (titulares["fecha"] >= pd.Timestamp(INI_TRAIN) - pd.Timedelta(days=30))
    & (titulares["fecha"] <= FIN_TEST)
].reset_index(drop=True)

if os.path.exists(ARCHIVO_PUNTAJES):
    titulares_periodo = pd.read_csv(ARCHIVO_PUNTAJES, parse_dates=["fecha"])
    print(f"Puntajes cargados de '{ARCHIVO_PUNTAJES}' ({len(titulares_periodo)} titulares).")
else:
    finbert = pipeline(
        "text-classification",
        model="ProsusAI/finbert",
        framework="pt",
        top_k=None,  # devuelve las probabilidades de las tres clases
        device=0 if torch.cuda.is_available() else -1,
    )
    inicio = time.perf_counter()
    salidas = finbert(titulares_periodo["headline"].tolist(), batch_size=32, truncation=True)
    print(f"{len(salidas)} titulares puntuados en {time.perf_counter() - inicio:.0f} s")

    probs = pd.DataFrame([{d["label"]: d["score"] for d in s} for s in salidas])
    titulares_periodo[["p_pos", "p_neg", "p_neu"]] = probs[["positive", "negative", "neutral"]].values
    titulares_periodo["puntaje"] = titulares_periodo["p_pos"] - titulares_periodo["p_neg"]
    titulares_periodo.to_csv(ARCHIVO_PUNTAJES, index=False)

    del finbert  # liberamos memoria antes de entrenar las LSTM

titulares_periodo["puntaje"].describe()

In [ ]:
# Los titulares más negativos y más positivos: ¿tiene sentido lo que dice FinBERT?
columnas = ["fecha", "puntaje", "headline"]
print("Más negativos:")
display(titulares_periodo.nsmallest(5, "puntaje")[columnas])
print("Más positivos:")
display(titulares_periodo.nlargest(5, "puntaje")[columnas])

### 4.2 De titulares a una serie diaria, sin mirar el futuro

Este es el paso donde es más fácil cometer un error grave sin darse cuenta: **usar una noticia antes
de que existiera**.

Nuestro pronóstico se hace al **cierre del día $t$**. Una noticia fechada el día $t$ pudo publicarse a
las 9 a. m. (disponible al cierre) o a las 8 p. m. (después del cierre, ya no disponible). Como en este
dataset casi todas las fechas vienen **sin hora**, no podemos distinguir esos casos. La regla
conservadora es:

> Una noticia con fecha $d$ se usa a partir del **primer día hábil estrictamente posterior a $d$**.

Así, una noticia del lunes se usa en el pronóstico hecho al cierre del martes, y una del sábado, en el
del lunes. Perdemos algo de inmediatez, pero garantizamos que ningún pronóstico use información que no
estaba disponible. Si en tu proyecto tienes la hora exacta de publicación, puedes afinar la regla
(noticias antes de las 4 p. m. hora de Nueva York → mismo día).

Con esa asignación construimos tres variables diarias:

| Variable | Definición | Para qué sirve |
|---|---|---|
| `sent_dia` | Promedio del puntaje de los titulares asignados al día (0 si no hubo) | Tono de las noticias recientes |
| `sent_5d` | Promedio del puntaje de todos los titulares de los últimos 5 días hábiles | Tono persistente, menos ruidoso |
| `log_n_titulares` | $\ln(1 + \text{número de titulares del día})$ | Intensidad de la cobertura: distingue "neutral" de "sin noticias" |

In [ ]:
dias_habiles = mercado.index

# Posición del primer día hábil estrictamente posterior a la fecha de cada noticia
posicion = np.searchsorted(dias_habiles.values, titulares_periodo["fecha"].values, side="right")
validas = posicion < len(dias_habiles)
asignadas = titulares_periodo.loc[validas].copy()
asignadas["dia_uso"] = dias_habiles[posicion[validas]]

por_dia = asignadas.groupby("dia_uso")["puntaje"].agg(suma="sum", n="count")
sentimiento = por_dia.reindex(dias_habiles, fill_value=0)

sentimiento["sent_dia"] = np.where(sentimiento["n"] > 0, sentimiento["suma"] / sentimiento["n"].clip(lower=1), 0.0)
suma_5d = sentimiento["suma"].rolling(5, min_periods=1).sum()
n_5d = sentimiento["n"].rolling(5, min_periods=1).sum()
sentimiento["sent_5d"] = np.where(n_5d > 0, suma_5d / n_5d.clip(lower=1), 0.0)
sentimiento["log_n_titulares"] = np.log1p(sentimiento["n"])

# Verificación: ninguna noticia se usa el mismo día de su fecha ni antes
assert (asignadas["dia_uso"] > asignadas["fecha"]).all()

sentimiento.loc["2020-03-02":"2020-03-13", ["n", "sent_dia", "sent_5d"]].round(3)

In [ ]:
fig, ax1 = plt.subplots(figsize=(11, 4))
serie = sentimiento.loc[INI_TRAIN:FIN_TEST, "sent_5d"].rolling(22).mean()
ax1.plot(serie.index, serie, color="tab:purple", label="Sentimiento (media móvil 22 días)")
ax1.axhline(0, color="gray", linewidth=0.8)
ax1.set_ylabel("Sentimiento FinBERT")
ax2 = ax1.twinx()
ax2.plot(mercado.loc[INI_TRAIN:FIN_TEST].index, mercado.loc[INI_TRAIN:FIN_TEST, "precio"],
         color="tab:blue", alpha=0.5, label="Precio")
ax2.set_ylabel("Precio (USD)")
ax1.set_title(f"{TICKER}: índice de sentimiento y precio")
fig.legend(loc="upper right")
plt.show()

<a id="sec5"></a>
## 5. Variables de entrada y conjuntos de datos

### 5.1 Variables de mercado y de sentimiento

Todas las variables del día $t$ se calculan con información disponible al cierre de $t$:

| Grupo | Variable | Definición |
|---|---|---|
| Mercado | `ret` | Retorno del día, $r_t$ (con la ventana de 22 días, la LSTM ve 22 rezagos) |
| Mercado | `log_ret_abs` | $\ln(\lvert r_t \rvert + 0.1)$: magnitud del movimiento, sin importar el signo |
| Mercado | `log_vol_5d`, `log_vol_22d` | Logaritmo de la desviación estándar de los retornos de los últimos 5 y 22 días (volatilidad histórica) |
| Mercado | `log_vol_rel` | $\ln(\text{volumen}_t) - \ln(\text{promedio de volumen de 22 días})$: volumen anormal |
| Sentimiento | `sent_dia`, `sent_5d`, `log_n_titulares` | Definidas en la sección 4.2 |

**¿Por qué logaritmos?** Por la misma razón que usamos $\ln RV$ como objetivo: las medidas de
volatilidad tienen colas derechas muy largas. En niveles, el −73% (logarítmico) del 9 de marzo de 2020
quedaría a unas 27 desviaciones estándar del promedio de entrenamiento, y una red neuronal extrapola muy
mal tan lejos de lo que vio al entrenar. En logaritmos ese mismo día queda a unas 4 desviaciones: sigue
siendo extremo, pero dentro de un rango razonable. Además, así las entradas y el objetivo están en la
misma escala (es la misma idea del modelo HAR en logaritmos, muy usado para volatilidad). El 0.1 dentro
del logaritmo evita $\ln 0$ en días con retorno exactamente cero.

In [ ]:
datos = mercado.join(sentimiento[["sent_dia", "sent_5d", "log_n_titulares"]])

datos["log_ret_abs"] = np.log(datos["ret"].abs() + 0.1)
datos["log_vol_5d"] = np.log(datos["ret"].rolling(5).std())
datos["log_vol_22d"] = np.log(datos["ret"].rolling(22).std())
datos["log_vol_rel"] = np.log(datos["volumen"]) - np.log(datos["volumen"].rolling(22).mean())

VARS_MERCADO = ["ret", "log_ret_abs", "log_vol_5d", "log_vol_22d", "log_vol_rel"]
VARS_SENTIMIENTO = ["sent_dia", "sent_5d", "log_n_titulares"]
VARS_TODAS = VARS_MERCADO + VARS_SENTIMIENTO

datos[VARS_TODAS + ["rv_futura"]].describe().T.round(3)

### 5.2 Un vistazo previo (solo con datos de entrenamiento)

Antes de entrenar nada, conviene preguntar si hay alguna relación simple entre el sentimiento y la
volatilidad futura. Lo miramos **solo en el periodo de entrenamiento**: mirar la prueba para decidir
cómo modelar sería hacer trampa.

Esperamos que la volatilidad histórica tenga una correlación alta con la futura (la volatilidad es
persistente). La pregunta interesante es si el sentimiento y la cobertura de noticias tienen alguna
correlación **adicional**.

In [ ]:
entrenamiento = datos.loc[INI_TRAIN:FIN_TRAIN]
correlaciones = entrenamiento[VARS_TODAS].corrwith(entrenamiento["log_rv_futura"], method="spearman")

fig, ax = plt.subplots(figsize=(7, 3.5))
correlaciones.sort_values().plot(kind="barh", ax=ax,
                                 color=["tab:purple" if v in VARS_SENTIMIENTO else "tab:blue" for v in correlaciones.sort_values().index])
ax.set_title("Correlación de Spearman con ln(RV futura), periodo de entrenamiento")
ax.axvline(0, color="gray")
plt.show()
correlaciones.round(3)

Una correlación baja no descarta que el sentimiento ayude: la LSTM puede aprovechar relaciones no
lineales o interacciones (por ejemplo, que las malas noticias importen más cuando la volatilidad ya es
alta). Tampoco una correlación alta garantizaría que ayude: puede ser información que ya está
contenida en la volatilidad histórica. Por eso la respuesta tiene que venir de comparar los modelos.

### 5.3 Particiones sin fugas de información

Cada muestra corresponde a un día de origen $t$ y se asigna al periodo donde cae $t$. Pero el objetivo
de $t$ usa los retornos de $t+1$ a $t+H$: las últimas $H$ muestras de entrenamiento tendrían un
objetivo que incluye retornos de validación. Las eliminamos (lo mismo al final de validación y de
prueba). A este margen se le llama **embargo**. Así, además, el último objetivo de prueba termina el
29 de mayo de 2020, respetando la fecha final del protocolo.

In [ ]:
datos = datos.dropna(subset=VARS_TODAS)
posiciones = pd.Series(np.arange(len(datos)), index=datos.index)


def indices_periodo(ini, fin):
    """Días de origen del periodo cuyo objetivo termina dentro del mismo periodo (embargo)."""
    en_periodo = (datos.index >= ini) & (datos.index <= fin)
    objetivo_dentro = datos["fecha_fin_objetivo"] <= pd.Timestamp(fin)
    return datos.index[en_periodo & objetivo_dentro.values]


idx_train = indices_periodo(INI_TRAIN, FIN_TRAIN)
idx_val = indices_periodo(INI_VAL, FIN_VAL)
idx_test = indices_periodo(INI_TEST, FIN_TEST)

for nombre, idx in [("Entrenamiento", idx_train), ("Validación", idx_val), ("Prueba", idx_test)]:
    print(f"{nombre:14s}: {len(idx):5d} muestras, {idx.min().date()} a {idx.max().date()}, "
          f"objetivo hasta {datos.loc[idx[-1], 'fecha_fin_objetivo'].date()}")

### 5.4 Estandarización y ventanas para la LSTM

- **Estandarización:** las variables tienen escalas muy distintas (retornos en %, conteos en
  logaritmo...). Restamos la media y dividimos por la desviación estándar **calculadas solo con el
  entrenamiento**, y aplicamos esa misma transformación a validación y prueba.
- **Ventanas:** para el día $t$, la LSTM recibe una matriz con las variables de los $L = 22$ días
  hábiles que terminan en $t$. Las ventanas pueden empezar en el periodo anterior (por ejemplo, la
  primera ventana de validación incluye días de diciembre de 2017): eso es legítimo, porque son datos
  del pasado.

In [ ]:
escalador = StandardScaler().fit(datos.loc[INI_TRAIN:FIN_TRAIN, VARS_TODAS])
X_escalado = pd.DataFrame(escalador.transform(datos[VARS_TODAS]), index=datos.index, columns=VARS_TODAS)

# El objetivo (ln RV) también se estandariza con estadísticos de entrenamiento
y_media = datos.loc[idx_train, "log_rv_futura"].mean()
y_desv = datos.loc[idx_train, "log_rv_futura"].std()


def construir_ventanas(indices, variables):
    """Devuelve X con forma (muestras, L, variables) e y estandarizado para los días de origen dados."""
    matriz = X_escalado[variables].to_numpy()
    pos = posiciones.loc[indices].to_numpy()
    if (pos < L - 1).any():
        raise ValueError("Hay días de origen sin historia suficiente para una ventana completa.")
    X = np.stack([matriz[p - L + 1 : p + 1] for p in pos])
    y = ((datos.loc[indices, "log_rv_futura"] - y_media) / y_desv).to_numpy()
    return X.astype("float32"), y.astype("float32")


X_tr, y_tr = construir_ventanas(idx_train, VARS_TODAS)
print(f"Ventanas de entrenamiento: {X_tr.shape}  (muestras, días, variables)")

<a id="sec6"></a>
## 6. Modelo 1: GARCH(1,1), la línea base

El modelo GARCH(1,1) describe la varianza condicional de mañana como una combinación de un nivel de
largo plazo, el *shock* de hoy y la varianza de hoy:

$$
\sigma_{t+1}^2 = \omega + \alpha \, \varepsilon_t^2 + \beta \, \sigma_t^2
$$

Es el estándar de la industria para volatilidad y, pese a su simplicidad, es una línea base difícil de
superar. Si una LSTM no le gana a GARCH, la complejidad adicional no se justifica.

Procedimiento, equivalente al de las LSTM:

1. Estimamos $\omega, \alpha, \beta$ **solo con los retornos de entrenamiento** (`first_obs` y
   `last_obs`; en `arch`, `last_obs` es exclusivo).
2. Con esos parámetros fijos, para cada día de origen $t$ de validación y prueba, el modelo actualiza
   $\sigma^2_t$ con los retornos observados hasta $t$ y pronostica la varianza de $t+1, \dots, t+H$.
3. El pronóstico comparable con $RV^{(H)}_t$ es la raíz del promedio de esas $H$ varianzas.

In [ ]:
modelo_garch = arch_model(mercado["ret"], mean="Constant", vol="GARCH", p=1, q=1, dist="normal")
ajuste_garch = modelo_garch.fit(first_obs=INI_TRAIN, last_obs=INI_VAL, disp="off")
print(f"Observaciones usadas en la estimación: {ajuste_garch.nobs}")
print(ajuste_garch.summary().tables[1])

alpha, beta = ajuste_garch.params["alpha[1]"], ajuste_garch.params["beta[1]"]
print(f"\nPersistencia α + β = {alpha + beta:.4f}  "
      f"(vida media de un shock ≈ {np.log(0.5) / np.log(alpha + beta):.0f} días)")

In [ ]:
pronostico_garch = ajuste_garch.forecast(horizon=H, start=INI_VAL, reindex=False)
vol_garch = np.sqrt(pronostico_garch.variance.mean(axis=1))  # índice: día de origen t

pred_garch_val = vol_garch.loc[idx_val]
pred_garch_test = vol_garch.loc[idx_test]
pred_garch_test.head()

<a id="sec7"></a>
## 7. Modelos 2 y 3: LSTM

### 7.1 Arquitectura

Usamos una red deliberadamente pequeña: tenemos ~1 500 muestras de entrenamiento, y una red grande se
sobreajustaría rápidamente.

```
Entrada (22 días × k variables)
   → LSTM(32)        resume la ventana en un vector de 32 números
   → Dropout(0.2)    apaga al azar 20% de las neuronas al entrenar (regularización)
   → Dense(16, relu)
   → Dense(1)        ln(RV) estandarizado
```

Los modelos 2 y 3 tienen **exactamente la misma arquitectura y el mismo entrenamiento**; solo cambia
$k$: 5 variables de mercado, o esas 5 más las 3 de sentimiento.

### 7.2 ¿Por qué entrenar con varias semillas?

Una red neuronal empieza con pesos aleatorios, y dos entrenamientos con semillas distintas dan
pronósticos distintos. Si entrenáramos cada modelo una sola vez, una diferencia entre el modelo 2 y el
3 podría deberse a la suerte de la semilla y no al sentimiento. Por eso entrenamos cada modelo
`N_SEMILLAS` veces y **promediamos los pronósticos** (un *ensamble*). Eso reduce la variabilidad y hace
la comparación mucho más justa.

El entrenamiento usa *early stopping*: se detiene cuando el error en validación deja de mejorar durante
15 épocas y se queda con los pesos de la mejor época.

### 7.3 De $\ln RV$ a $RV$: una corrección necesaria

La red pronostica $\ln RV$. Si simplemente aplicamos $\exp(\cdot)$ al pronóstico, obtenemos algo
cercano a la **mediana** de $RV$, no a su **media**, y la mediana de una variable con cola derecha
larga es menor que la media. El resultado sería subestimar la volatilidad de forma sistemática, que es
justo el error que QLIKE más castiga (y el que más le preocupa a un gestor de riesgos).

La corrección estándar supone que el error en logaritmos es aproximadamente normal con varianza $s^2$,
y multiplica el pronóstico por $e^{s^2/2}$:

$$
\widehat{RV} = \exp\left(\widehat{\ln RV} + \tfrac{s^2}{2}\right)
$$

Estimamos $s^2$ con los residuos del ensamble **en validación**, nunca en prueba. GARCH no necesita
esta corrección porque pronostica la varianza directamente.

In [ ]:
def construir_lstm(n_variables):
    modelo = keras.Sequential([
        layers.Input(shape=(L, n_variables)),
        layers.LSTM(32),
        layers.Dropout(0.2),
        layers.Dense(16, activation="relu"),
        layers.Dense(1),
    ])
    modelo.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="mse")
    return modelo


def entrenar_ensamble(variables, nombre):
    """Entrena N_SEMILLAS LSTM con las variables dadas y devuelve el pronóstico promedio de RV."""
    X_tr, y_tr = construir_ventanas(idx_train, variables)
    X_va, y_va = construir_ventanas(idx_val, variables)
    X_te, _ = construir_ventanas(idx_test, variables)

    preds_val, preds_test, historiales = [], [], []
    for i in range(N_SEMILLAS):
        keras.utils.set_random_seed(SEED + i)
        modelo = construir_lstm(len(variables))
        parada = keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)
        hist = modelo.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=200, batch_size=32,
                          callbacks=[parada], verbose=0)
        historiales.append(hist.history)
        preds_val.append(modelo.predict(X_va, verbose=0).ravel())
        preds_test.append(modelo.predict(X_te, verbose=0).ravel())
        print(f"  {nombre} · semilla {i + 1}/{N_SEMILLAS}: {len(hist.history['loss'])} épocas, "
              f"mejor val_loss = {min(hist.history['val_loss']):.4f}")

    # Promediamos las semillas en escala logarítmica y deshacemos la estandarización
    log_val = np.mean(preds_val, axis=0) * y_desv + y_media
    log_test = np.mean(preds_test, axis=0) * y_desv + y_media

    # Corrección lognormal (sección 7.3), estimada solo con los residuos de validación
    s2 = np.var(datos.loc[idx_val, "log_rv_futura"].to_numpy() - log_val)
    print(f"  {nombre} · varianza de residuos en validación s² = {s2:.3f} "
          f"→ factor de corrección e^(s²/2) = {np.exp(s2 / 2):.3f}")

    return (pd.Series(np.exp(log_val + s2 / 2), index=idx_val),
            pd.Series(np.exp(log_test + s2 / 2), index=idx_test),
            historiales)

In [ ]:
inicio = time.perf_counter()
print("Modelo 2: LSTM solo con variables de mercado")
pred_lstm_m_val, pred_lstm_m_test, hist_m = entrenar_ensamble(VARS_MERCADO, "LSTM mercado")
print("Modelo 3: LSTM con variables de mercado + sentimiento")
pred_lstm_ms_val, pred_lstm_ms_test, hist_ms = entrenar_ensamble(VARS_TODAS, "LSTM mercado+sent")
print(f"\nTiempo total de entrenamiento: {(time.perf_counter() - inicio) / 60:.1f} min")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
for ax, historiales, titulo in [(axes[0], hist_m, "LSTM mercado"), (axes[1], hist_ms, "LSTM mercado + sentimiento")]:
    for h in historiales:
        ax.plot(h["loss"], color="tab:blue", alpha=0.4)
        ax.plot(h["val_loss"], color="tab:orange", alpha=0.6)
    ax.set_title(titulo)
    ax.set_xlabel("Época")
axes[0].set_ylabel("MSE (ln RV estandarizado)")
axes[0].plot([], color="tab:blue", label="Entrenamiento")
axes[0].plot([], color="tab:orange", label="Validación")
axes[0].legend()
plt.tight_layout()
plt.show()

**Cómo leer estas curvas.** El objetivo está estandarizado, así que un `val_loss` cercano a 1
significa que la red **no pronostica mucho mejor que el promedio histórico**. Antes de concluir que la
LSTM está mal entrenada, compárala con alternativas simples. En la ejecución de referencia de este
ejercicio (OXY, $H = 5$), una regresión lineal con las mismas variables de mercado obtuvo en validación
un MSE de ~1,12, y un pronóstico constante, ~1,13: todos los modelos quedan en el mismo rango. La razón
es que la volatilidad realizada de 5 días, calculada con solo 5 retornos diarios, es una medida muy
ruidosa, y buena parte de su variación es impredecible con cualquier modelo.

Observa también que la mejor época suele llegar pronto: con ~1 500 muestras ruidosas, la red empieza a
memorizar el entrenamiento muy rápido, y el *early stopping* es lo que evita que ese sobreajuste llegue
a la prueba.

<a id="sec8"></a>
## 8. Evaluación en el periodo de prueba

### 8.1 Métricas

| Métrica | Fórmula | Qué castiga |
|---|---|---|
| **RMSE** | $\sqrt{\text{prom}(RV - \hat\sigma)^2}$ | Errores grandes (muy sensible a los días de crisis) |
| **MAE** | $\text{prom}\lvert RV - \hat\sigma \rvert$ | Todos los errores por igual |
| **QLIKE** | $\text{prom}\left(\frac{RV^2}{\hat\sigma^2} - \ln\frac{RV^2}{\hat\sigma^2} - 1\right)$ | Sobre todo **subestimar** la volatilidad |

QLIKE es la métrica preferida en la literatura de volatilidad (Patton, 2011): ordena los modelos
correctamente aunque la volatilidad realizada sea una medida ruidosa de la verdadera, y castiga más
subestimar el riesgo que sobreestimarlo, que es justo lo que le preocupa a un gestor de riesgos. En las
tres métricas, **menor es mejor**. QLIKE vale 0 si el pronóstico es perfecto.

In [ ]:
def perdida_qlike(rv, pred):
    cociente = (rv / pred) ** 2
    return cociente - np.log(cociente) - 1


def metricas(rv, pred):
    return {
        "RMSE": np.sqrt(np.mean((rv - pred) ** 2)),
        "MAE": np.mean(np.abs(rv - pred)),
        "QLIKE": np.mean(perdida_qlike(rv, pred)),
    }


PRONOSTICOS_TEST = {
    "GARCH(1,1)": pred_garch_test,
    "LSTM mercado": pred_lstm_m_test,
    "LSTM mercado + sentimiento": pred_lstm_ms_test,
}
PRONOSTICOS_VAL = {
    "GARCH(1,1)": pred_garch_val,
    "LSTM mercado": pred_lstm_m_val,
    "LSTM mercado + sentimiento": pred_lstm_ms_val,
}

rv_val = datos.loc[idx_val, "rv_futura"]
rv_test = datos.loc[idx_test, "rv_futura"]

tabla_val = pd.DataFrame({m: metricas(rv_val, p) for m, p in PRONOSTICOS_VAL.items()}).T
tabla_test = pd.DataFrame({m: metricas(rv_test, p) for m, p in PRONOSTICOS_TEST.items()}).T

print("Validación (2018):")
display(tabla_val.round(4))
print(f"Prueba ({INI_TEST} a {FIN_TEST}):")
display(tabla_test.round(4))

### 8.2 Calma contra crisis

Un promedio sobre todo el periodo de prueba puede esconder comportamientos muy distintos. Separamos la
prueba en dos subperiodos:

- **Calma:** enero de 2019 a febrero de 2020.
- **Crisis:** marzo a mayo de 2020 (desplome del petróleo y COVID).

In [ ]:
SUBPERIODOS = {
    "Calma (2019-01 a 2020-02)": (INI_TEST, "2020-02-28"),
    "Crisis (2020-03 a 2020-05)": ("2020-03-02", FIN_TEST),
}

filas = []
for subperiodo, (ini, fin) in SUBPERIODOS.items():
    mascara = (idx_test >= ini) & (idx_test <= fin)
    for modelo, pred in PRONOSTICOS_TEST.items():
        filas.append({"subperiodo": subperiodo, "modelo": modelo, "n": mascara.sum(),
                      **metricas(rv_test[mascara], pred[mascara])})

tabla_sub = pd.DataFrame(filas).set_index(["subperiodo", "modelo"])
tabla_sub.round(4)

### 8.3 ¿La diferencia es significativa? Prueba de Diebold-Mariano

Que un modelo tenga un QLIKE menor en esta muestra no demuestra que sea mejor: con ~340 días de prueba
(y objetivos que se solapan, porque dos días consecutivos comparten 4 de sus 5 retornos futuros), una
diferencia pequeña puede ser pura casualidad.

La **prueba de Diebold-Mariano** (1995) compara las pérdidas de dos modelos día a día,
$d_t = \text{pérdida}_A(t) - \text{pérdida}_B(t)$, y pregunta si el promedio de $d_t$ es
significativamente distinto de cero:

- $H_0$: los dos modelos pronostican igual de bien.
- El estadístico se construye como una prueba *t* sobre $\bar d$, pero usando una varianza que tiene en
  cuenta la autocorrelación de $d_t$ (varianza de Newey-West con $H-1$ rezagos, por el solapamiento) y
  la corrección para muestras pequeñas de Harvey, Leybourne y Newbold (1997).
- **Estadístico positivo** → el modelo A tiene más pérdida → **B es mejor**.

Para nuestra pregunta, A = LSTM de mercado y B = LSTM con sentimiento, y la hipótesis relevante es
unilateral: *¿el sentimiento reduce la pérdida?* Reportamos ambos valores p.

In [ ]:
def prueba_diebold_mariano(perdida_a, perdida_b, h):
    """Prueba DM con varianza Newey-West (h-1 rezagos) y corrección de Harvey-Leybourne-Newbold.

    Devuelve (estadístico, p bilateral, p unilateral para H1: B es mejor que A).
    """
    d = np.asarray(perdida_a) - np.asarray(perdida_b)
    n = len(d)
    d_centrado = d - d.mean()
    varianza = np.sum(d_centrado ** 2) / n
    for k in range(1, h):
        autocov = np.sum(d_centrado[k:] * d_centrado[:-k]) / n
        varianza += 2 * (1 - k / h) * autocov
    estadistico = d.mean() / np.sqrt(varianza / n)
    estadistico *= np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)  # corrección HLN
    p_bilateral = 2 * stats.t.sf(abs(estadistico), df=n - 1)
    p_unilateral = stats.t.sf(estadistico, df=n - 1)
    return estadistico, p_bilateral, p_unilateral


COMPARACIONES = [
    ("LSTM mercado", "LSTM mercado + sentimiento"),  # la pregunta de fondo
    ("GARCH(1,1)", "LSTM mercado"),
    ("GARCH(1,1)", "LSTM mercado + sentimiento"),
]

filas = []
for a, b in COMPARACIONES:
    for nombre_perdida, funcion in [("QLIKE", perdida_qlike), ("Error²", lambda rv, p: (rv - p) ** 2)]:
        est, p_bi, p_uni = prueba_diebold_mariano(funcion(rv_test, PRONOSTICOS_TEST[a]),
                                                  funcion(rv_test, PRONOSTICOS_TEST[b]), H)
        filas.append({"A": a, "B": b, "pérdida": nombre_perdida, "estadístico DM": est,
                      "p bilateral": p_bi, "p unilateral (B mejor)": p_uni})

tabla_dm = pd.DataFrame(filas)
tabla_dm.round(4)

### 8.4 Los pronósticos en el tiempo

In [ ]:
anualizar = np.sqrt(252)
estilos = {"GARCH(1,1)": ("tab:green", "-"), "LSTM mercado": ("tab:blue", "-"),
           "LSTM mercado + sentimiento": ("tab:red", "--")}

fig, axes = plt.subplots(2, 1, figsize=(12, 8))
for ax, (ini, fin), titulo in [(axes[0], (INI_TEST, FIN_TEST), "Todo el periodo de prueba"),
                              (axes[1], ("2020-02-01", FIN_TEST), "Zoom: febrero a mayo de 2020")]:
    ax.plot(rv_test.loc[ini:fin] * anualizar, color="black", linewidth=1, label="RV realizada")
    for modelo, pred in PRONOSTICOS_TEST.items():
        color, linea = estilos[modelo]
        ax.plot(pred.loc[ini:fin] * anualizar, color=color, linestyle=linea, label=modelo)
    ax.set_title(titulo)
    ax.set_ylabel(f"Volatilidad {H} días (anualizada, %)")
axes[0].legend()
plt.tight_layout()
plt.show()

Mira con detalle el zoom de marzo de 2020. Preguntas para discutir en clase:

- ¿Qué modelo reacciona más rápido después del 9 de marzo? ¿Alguno lo anticipó?
- Los tres modelos usan información hasta el día $t$. ¿Por qué ninguno puede pronosticar bien el
  primer día del choque?
- ¿Cuál vuelve más rápido a niveles normales en abril y mayo? ¿Es eso una virtud o un defecto?

<a id="sec9"></a>
## 9. Discusión: ¿el sentimiento agrega poder predictivo?

La siguiente celda redacta la respuesta **a partir de los resultados que obtuviste** en esta ejecución.
No asumimos de antemano qué modelo gana: los resultados pueden cambiar si modificas el horizonte, la
empresa, el número de semillas o la construcción del índice.

In [ ]:
NIVEL = 0.05
qlike = tabla_test["QLIKE"]
dm_principal = tabla_dm[(tabla_dm["A"] == "LSTM mercado") & (tabla_dm["pérdida"] == "QLIKE")].iloc[0]
cambio_pct = 100 * (qlike["LSTM mercado + sentimiento"] / qlike["LSTM mercado"] - 1)

texto = [f"**Mejor modelo en prueba según QLIKE:** {qlike.idxmin()} ({qlike.min():.4f}).", ""]

if cambio_pct < 0:
    texto.append(f"Agregar el sentimiento **redujo** el QLIKE de la LSTM en {abs(cambio_pct):.1f}% "
                 f"({qlike['LSTM mercado']:.4f} → {qlike['LSTM mercado + sentimiento']:.4f}).")
else:
    texto.append(f"Agregar el sentimiento **no redujo** el QLIKE de la LSTM: cambió {cambio_pct:+.1f}% "
                 f"({qlike['LSTM mercado']:.4f} → {qlike['LSTM mercado + sentimiento']:.4f}).")

def veredicto_dm(fila, nombre_a, nombre_b):
    """Traduce una fila de la tabla DM a una frase: B mejor, A mejor o sin diferencia significativa."""
    if fila["p unilateral (B mejor)"] < NIVEL:
        return (f"la prueba de Diebold-Mariano indica que **{nombre_b} es significativamente mejor** "
                f"(p unilateral = {fila['p unilateral (B mejor)']:.3f})")
    if fila["p bilateral"] < NIVEL and fila["estadístico DM"] < 0:
        return (f"la prueba de Diebold-Mariano indica que **{nombre_b} es significativamente peor** que "
                f"{nombre_a} (p bilateral = {fila['p bilateral']:.3f})")
    return (f"la prueba de Diebold-Mariano **no encuentra una diferencia significativa** "
            f"(p bilateral = {fila['p bilateral']:.3f}); la diferencia observada es compatible con el azar")


veredicto = veredicto_dm(dm_principal, "la LSTM de mercado", "el modelo con sentimiento")
texto.append(f"Con pérdida QLIKE, {veredicto}.")
if dm_principal["p unilateral (B mejor)"] < NIVEL:
    texto.append("**Conclusión:** en este periodo hay evidencia estadística de que el sentimiento agrega poder "
                 "predictivo.")
else:
    texto.append("**Conclusión:** con esta muestra y esta forma de construir el índice, **no hay evidencia de que "
                 "el sentimiento agregue poder predictivo**. Eso no demuestra que el sentimiento sea inútil: "
                 "revisa las limitaciones de la sección 9.1 antes de generalizar.")

crisis = tabla_sub.loc["Crisis (2020-03 a 2020-05)", "QLIKE"]
calma = tabla_sub.loc["Calma (2019-01 a 2020-02)", "QLIKE"]
texto += ["",
          f"**Por subperiodo (QLIKE):** en calma, el sentimiento cambió la pérdida de la LSTM de "
          f"{calma['LSTM mercado']:.4f} a {calma['LSTM mercado + sentimiento']:.4f}; en crisis, de "
          f"{crisis['LSTM mercado']:.4f} a {crisis['LSTM mercado + sentimiento']:.4f}."]

texto.append("**Frente a la línea base GARCH(1,1):**")
for lstm in ["LSTM mercado", "LSTM mercado + sentimiento"]:
    fila = tabla_dm[(tabla_dm["A"] == "GARCH(1,1)") & (tabla_dm["B"] == lstm)
                    & (tabla_dm["pérdida"] == "QLIKE")].iloc[0]
    texto.append(f"- {lstm} (QLIKE {qlike[lstm]:.4f} contra {qlike['GARCH(1,1)']:.4f}): "
                 f"{veredicto_dm(fila, 'GARCH(1,1)', lstm)}.")

display(Markdown("\n\n".join(texto)))

### 9.1 Limitaciones que debes discutir en tu informe

Cualquiera que sea el resultado, una conclusión honesta tiene que reconocer estas limitaciones:

1. **Calidad del índice de sentimiento.** La cobertura de noticias es irregular y muchos titulares no
   son realmente sobre la empresa. Un índice ruidoso puede esconder una señal que sí existe.
2. **FinBERT fue entrenado con otro tipo de texto** (frases de noticias etiquetadas por su efecto en el
   precio, no en la volatilidad). El *tono* de una noticia y la *incertidumbre* que genera no son lo
   mismo: una noticia muy positiva e inesperada también mueve la volatilidad.
3. **Regla de un día de rezago.** Por no tener la hora de publicación, usamos cada noticia un día
   después. Si el efecto del sentimiento se concentra en las primeras horas, lo estamos perdiendo.
4. **Una sola empresa y un solo periodo de prueba.** ~340 días con objetivos solapados equivalen a
   bastante menos observaciones independientes. Un resultado con OXY no se generaliza automáticamente.
5. **GARCH con parámetros fijos.** No reestimamos GARCH durante la prueba; con reestimación periódica
   podría adaptarse mejor al cambio de régimen de 2020 (lo mismo aplica a reentrenar las LSTM).
6. **La volatilidad realizada es solo una aproximación** de la volatilidad verdadera, construida con
   5 retornos diarios. Con datos intradía la medición sería mucho más precisa.

<a id="sec10"></a>
## 10. Actividades y entregables

### Actividades

**A. Replicación (obligatoria).** Ejecuta el cuaderno completo y reporta la tabla de métricas de prueba,
la tabla por subperiodos y la prueba de Diebold-Mariano. Explica con tus palabras qué significa el
resultado de la comparación LSTM mercado contra LSTM mercado + sentimiento.

**B. Sensibilidad al horizonte (obligatoria).** Repite el ejercicio con `H = 1` y `H = 22`. ¿El
sentimiento ayuda más a corto o a largo plazo? ¿Cambia el ranking frente a GARCH?

**C. Mejorar el índice de sentimiento (elige una).**

1. Quédate solo con los titulares que mencionan explícitamente a la empresa (por ejemplo, que contengan
   "Occidental" u "OXY") y descarta los que listan muchos tickers.
2. Construye un índice con decaimiento exponencial en lugar de la media de 5 días
   (`ewm(halflife=...)`), para que las noticias viejas pesen menos sin desaparecer de golpe.
3. Separa las noticias positivas de las negativas en dos variables (`sent_pos`, `sent_neg`): la
   literatura sugiere que las malas noticias mueven más la volatilidad que las buenas.

**D. Mejorar la línea base (opcional).** Prueba un GJR-GARCH (`o=1` en `arch_model`, que permite que los
choques negativos aumenten más la volatilidad) o una distribución *t* de Student (`dist="t"`). ¿Sigue
siendo difícil de superar?

**E. Otra empresa (opcional).** Usa la función `cobertura` para elegir otra empresa con noticias
suficientes en los tres periodos (por ejemplo KR, BLK o MDT tienen buena cobertura) y repite el
análisis. ¿La conclusión se mantiene?

### Entregables

1. El cuaderno ejecutado, con tus modificaciones de las actividades B y C.
2. Un informe de máximo dos páginas que responda la pregunta de fondo: *¿el sentimiento agrega poder
   predictivo al pronóstico de volatilidad de OXY?* Debe incluir la evidencia (métricas y
   Diebold-Mariano), la interpretación del periodo de crisis y al menos tres limitaciones de la
   sección 9.1 con una propuesta concreta para cada una.

### Criterios de evaluación

| Criterio | Peso |
|---|---|
| Replicación correcta y sin fugas de información (actividad A) | 25% |
| Análisis de sensibilidad al horizonte (actividad B) | 20% |
| Mejora del índice de sentimiento, bien justificada (actividad C) | 25% |
| Informe: conclusión respaldada por la prueba estadística, no solo por las métricas | 20% |
| Discusión de limitaciones con propuestas concretas | 10% |

### Referencias

- Diebold, F. X. y Mariano, R. S. (1995). Comparing predictive accuracy. *Journal of Business &
  Economic Statistics*, 13(3), 253-263.
- Harvey, D., Leybourne, S. y Newbold, P. (1997). Testing the equality of prediction mean squared
  errors. *International Journal of Forecasting*, 13(2), 281-291.
- Patton, A. J. (2011). Volatility forecast comparison using imperfect volatility proxies. *Journal of
  Econometrics*, 160(1), 246-256.
- Araci, D. (2019). FinBERT: Financial sentiment analysis with pre-trained language models.
  *arXiv:1908.10063*.
- Aenlle, M. (2020). *Daily Financial News for 6000+ Stocks* [dataset]. Kaggle.